# Jiaju — calibration, predictor and risk stability

**Owner:** Jiaju. **Model:** LightGBM `no_dc` of run `aki_ml_v1_28e0b1fb83`, Platt calibration (fixed; do not
retrain). This is notebook 03 Part 5: check whether risk estimates and predictors stay stable across ICU time and
across the 6h / 12h / 24h / 48h horizons, on **validation** patients.

| Part | Task | Output (in `artifacts/handoff/jiaju/`) |
|---|---|---|
| 0 | Reproduce the stored validation risks with the shipped model (must pass first) | assertion only |
| 5.1 | Calibration by ICU-time segment and horizon | `calibration_by_segment.csv`, figure |
| 5.2 | Predictor (TreeSHAP) stability across segments, horizons and bootstrap resamples | `shap_stability.csv`, figure |
| 5.3 | Feature-distribution and missingness shift over ICU time | `feature_shift.csv` |
| 5.4 | Risk volatility between updates (hourly and event-driven) | `risk_volatility.csv`, figure |
| 5.5 | Monotonic consistency across horizons (6h <= 12h <= 24h <= 48h) | `horizon_monotonicity.csv` |
| 5.6 | Reliability summary per horizon x segment | `reliability_summary.csv` |

**Data in this hand-off:** validation snapshots with features and labels, hourly predictions (train OOF and
validation; train OOF only for comparing fold stability in 5.1), event-driven replay predictions for validation,
and the fitted model. Training features and the test split are not shipped.

**Ground rules**
1. Validation only for every analysis (train OOF predictions may be used where stated). No test data.
2. Do not edit `src/aki_ml/` (its hash is recorded by notebook 02). Put reusable code in `src/aki_eval/stability.py`.
3. Outputs are aggregate only - never patient-level rows, IDs or timestamps. Explanations describe associations,
   not causes.
4. Hand back through a GitHub pull request: this notebook (outputs cleared of patient-level data),
   `src/aki_eval/stability.py`, and the files listed above.

Execution notes: all analyses run locally. Fixed seed 20261102; 200 calibration and 100 SHAP patient bootstrap draws. Event updates use supplied hourly_snapshot_hour segments. Volatility reports all consecutive updates plus an eligible-update sensitivity; hourly gaps are excluded. Consistent same-second event bundles are coalesced and n_events summed; conflicting states stop execution. Reversal fractions exclude right-censored crossings unless reversal is observed. No new reliability cutoffs are imposed.


In [ ]:
import hashlib
import json
import sys
from importlib import metadata
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "configs" / "ml.yaml").exists() and (path / "pipeline").exists():
            return path
    raise FileNotFoundError("Run this notebook from inside the AIForHealth-AKIEarlyWarning repository.")


# Standalone delivery: locate source/data without copying patient-level files.
import os
DELIVERY_ROOT = Path(os.environ["AKI_DELIVERY_ROOT"]).resolve() if "AKI_DELIVERY_ROOT" in os.environ else next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "aki_eval").is_dir()), Path.cwd())
if (DELIVERY_ROOT / "src" / "aki_eval").is_dir():
    sys.path.insert(0, str(DELIVERY_ROOT / "src"))
REPO_ROOT = Path(os.environ.get("AKI_REPO_ROOT", "C:/AKI_handoff_v1/AIForHealth-AKIEarlyWarning")).resolve()
if not (REPO_ROOT / "configs" / "ml.yaml").exists():
    REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))

from aki_ml import load_feature_matrix, logit

RUN_ID = "aki_ml_v1_28e0b1fb83"
FAMILY, FEATURE_SET = "lightgbm", "no_dc"
HORIZONS = [6, 12, 24, 48]
SEGMENTS = [(1, 3), (4, 6), (7, 12), (13, 18), (19, 24), (25, 36), (37, 48), (49, 60), (61, 72)]   # as Demo I
SEED = 20261102
DATA_ROOT = (REPO_ROOT / ".." / "Cleaned").resolve()
PRED_PATH = REPO_ROOT / "artifacts" / "predictions" / f"{RUN_ID}.parquet"
REPLAY_PATH = REPO_ROOT / "artifacts" / "predictions" / f"{RUN_ID}_event_replay.parquet"
MODEL_DIR = REPO_ROOT / "artifacts" / "models" / RUN_ID / f"{FAMILY}__{FEATURE_SET}"
POLICY = json.loads((REPO_ROOT / "artifacts" / "demo_i" / "policy.json").read_text(encoding="utf-8"))
RUN_MANIFEST = json.loads((REPO_ROOT / "artifacts" / "models" / RUN_ID / "run_manifest.json").read_text(encoding="utf-8"))
# The hand-off copy ships validation rows only; the full local dataset is filtered to validation on read.
SNAP_PATH = next(p for p in [DATA_ROOT / "datasets" / "snapshot_dataset_validation.parquet",
                             DATA_ROOT / "datasets" / "snapshot_dataset.parquet"] if p.exists())
FEATURE_DICT = pd.read_csv(DATA_ROOT / "datasets" / "feature_dictionary.csv")
OUT = DELIVERY_ROOT / "artifacts" / "handoff" / "jiaju"
OUT.mkdir(parents=True, exist_ok=True)

PINNED = {"numpy": "2.1.3", "pandas": "2.2.3", "pyarrow": "19.0.0", "scikit-learn": "1.6.1", "lightgbm": "4.7.0"}
wrong = {k: metadata.version(k) for k, v in PINNED.items() if metadata.version(k) != v}
assert not wrong, f"package versions differ from the training environment: {wrong}"
print("Repository:", REPO_ROOT)
print("Data:", SNAP_PATH.relative_to(DATA_ROOT.parent))
print("Reliability criteria recorded by notebook 02 for notebook 03:", RUN_MANIFEST["reliability_criteria_for_notebook_03"])
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)
from aki_eval import stability as st


## Part 0 — Reproduce the stored validation risks (must pass before any new work)

Loads the pickled pipeline (hash-checked against `policy.json`), scores every validation snapshot for all four
horizons, and compares with the risks that notebook 02 stored. Also re-derives the validation AUROC per horizon
from `artifacts/demo_i/validation_discrimination.csv`. If this fails, the environment or data copy is wrong.

In [ ]:
pipeline_file = MODEL_DIR / "pipeline.joblib"
assert hashlib.sha256(pipeline_file.read_bytes()).hexdigest() == POLICY["model"]["pipeline_sha256"], "model file changed"
pipe = joblib.load(pipeline_file)
FEATURES = pipe.feature_names

VAL_FILTER = [("split", "=", "validation")]
keys = pq.read_table(SNAP_PATH, columns=["patient_id", "hours_since_icu"], filters=VAL_FILTER).to_pandas()
keys["patient_id"] = keys["patient_id"].astype(str)
keys["hours_since_icu"] = keys["hours_since_icu"].round().astype(int)
X_val = load_feature_matrix(SNAP_PATH, FEATURES, filters=VAL_FILTER)     # float32, rows aligned with `keys`
print(f"validation: {keys['patient_id'].nunique():,} patients, {len(keys):,} snapshots x {len(FEATURES)} features")

pred = pq.read_table(PRED_PATH, filters=[("family", "=", FAMILY), ("feature_set", "=", FEATURE_SET),
                                         ("prediction_set", "=", "validation")],
                     columns=["horizon_h", "patient_id", "hours_since_icu", "snapshot_time", "creatinine_aki_onset_time",
                              "eligible", "label", "risk_raw", "risk_platt", "risk_isotonic"]).to_pandas()
pred["patient_id"] = pred["patient_id"].astype(str)
assert pred["horizon_h"].isin(HORIZONS).all()

disc = pd.read_csv(REPO_ROOT / "artifacts" / "demo_i" / "validation_discrimination.csv")
part0_evidence = []
for h in HORIZONS:
    stored = keys.merge(pred[pred["horizon_h"] == h], on=["patient_id", "hours_since_icu"], how="left", validate="1:1")
    assert stored["risk_platt"].notna().all(), f"{h}h: snapshots without a stored prediction"
    risk = pipe.predict(X_val, h, "platt")
    diff = np.abs(risk - stored["risk_platt"].to_numpy(np.float64)).max()
    e = stored["eligible"].to_numpy()
    auroc = roc_auc_score(stored.loc[e, "label"].astype(int), risk[e])
    ref = disc.query("horizon_h == @h and scope == 'hours 1-72' and calibration == 'platt'")["auroc"].item()
    assert diff < 1e-6 and abs(auroc - ref) < 1e-9, f"{h}h: max |risk diff| {diff:.2e}, AUROC {auroc:.6f} vs {ref:.6f}"
    print(f"{h:>2}h: stored risks reproduced (max |diff| {diff:.1e}), validation AUROC {auroc:.3f}")
    part0_evidence.append(dict(horizon_h=h, max_risk_difference=float(diff), auroc=float(auroc), reference_auroc=float(ref), passed=True))
pd.DataFrame(part0_evidence).to_csv(OUT / "part0_reproduction.csv", index=False)


### Helpers available to every part

- `pred`: hourly validation predictions, all horizons (`risk_raw`, `risk_platt`, `risk_isotonic`, `eligible`, `label`).
- `X_val` / `keys`: validation feature matrix and its (patient_id, hours_since_icu) rows.
- `pipe`: fitted pipeline. `pipe.heads[h]` is a LightGBM Booster; `pipe.n_rounds[h]` the rounds used;
  `pipe.calibrators[h]["platt"].slope_` the Platt slope.
- `tree_shap(X, h)` below: TreeSHAP contributions on the calibrated log-odds scale (same as the dashboard drivers).
  Memory: about 2 KB per row; use a patient-level sample (e.g. 1,500 patients) rather than all rows.
- `load_replay()`: event-driven validation predictions (one row per input event time and horizon).

In [ ]:
def tree_shap(X, h):
    """TreeSHAP contributions (n x features) on the Platt log-odds scale; checked to add up to the score."""
    raw = pipe.raw_score(X, h)
    contrib = pipe.heads[h].predict(X, num_iteration=pipe.n_rounds[h], pred_contrib=True, num_threads=4)
    assert np.allclose(contrib.sum(axis=1), logit(raw), atol=1e-3), "TreeSHAP does not add up to the score"
    return contrib[:, :-1] * pipe.calibrators[h]["platt"].slope_


def load_replay(horizons=HORIZONS):
    t = pq.read_table(REPLAY_PATH, filters=[("family", "=", FAMILY), ("feature_set", "=", FEATURE_SET),
                                            ("horizon_h", "in", horizons)],
                      columns=["horizon_h", "patient_id", "event_time", "hours_since_icu", "hourly_snapshot_hour",
                               "n_events", "eligible", "label", "risk_platt"])
    d = t.to_pandas()
    d["patient_id"] = d["patient_id"].astype(str)
    return d


def segment_of(hours):
    out = pd.Series(pd.NA, index=getattr(hours, "index", None), dtype="object")
    for lo, hi in SEGMENTS:
        out[(hours >= lo) & (hours <= hi)] = f"{lo}-{hi}h"
    return out

## 5.1 Calibration by ICU-time segment and horizon

For each horizon x segment (eligible rows): rows, positives, prevalence, AUROC, AUPRC, Brier, calibration slope and
intercept (logistic recalibration on logit(risk), as in Demo I), observed/expected ratio, for Platt (primary) and
isotonic (sensitivity). Patient-level bootstrap 95% CIs (>= 200 resamples) for AUROC and slope. Also report the
spread of OOF fold metrics on train (`prediction_set == "train_oof"`, column `cv_fold`) per horizon - one of the
notebook 02 reliability criteria. Figure: calibration slope and O/E by segment, one line per horizon, with the
[0.8, 1.25] band.

In [ ]:
oof = pq.read_table(PRED_PATH,
    filters=[("family", "=", FAMILY), ("feature_set", "=", FEATURE_SET), ("prediction_set", "=", "train_oof")],
    columns=["horizon_h", "cv_fold", "patient_id", "eligible", "label", "risk_raw", "risk_platt", "risk_isotonic"]).to_pandas()
calibration_table = st.calibration(pred, oof, OUT, n_boot=200)
del oof


## 5.2 Predictor stability (TreeSHAP)

On a fixed patient-level sample of validation patients (seed `SEED`):
- mean |SHAP| per feature for each horizon x segment; top-20 features per cell;
- rank agreement between cells (Spearman of mean |SHAP|, and top-20 overlap / Jaccard) across adjacent segments
  and across horizons;
- bootstrap over patients (>= 100 resamples): how often each top-20 feature stays in the top 20;
- sign consistency: share of rows where a feature's SHAP sign agrees with its overall direction.
Flag features whose importance changes rank by > 10 places between adjacent segments.

In [ ]:
shap_table, shap_agreement = st.shap_stability(X_val, keys, FEATURES, tree_shap, OUT, n_patients=1500, n_boot=100)


## 5.3 Feature-distribution and missingness shift over ICU time

Per feature and segment: missing fraction, median and IQR (validation). Compare with the train missing fractions
in `feature_dictionary.csv` (`missing_fraction_train`). Summarise shift with a standardised difference or
population stability index between the first segment and each later one, and list the 20 most-shifted features
together with their 5.2 importance (shift matters most for important features).

In [ ]:
shift_table = st.feature_shift(X_val, keys, FEATURES, FEATURE_DICT, OUT)
importance = shap_table.groupby(["feature", "segment"]).mean_abs_shap.mean().rename("mean_abs_shap_across_horizons").reset_index()
shift_table = shift_table.merge(importance, on=["feature", "segment"], how="left", validate="1:1")
shift_table.to_csv(OUT / "feature_shift.csv", index=False)
shift_table.query("segment != '1-3h'").sort_values("abs_standardized_difference", ascending=False).groupby("segment", sort=False).head(20).to_csv(OUT / "feature_shift_top20.csv", index=False)


## 5.4 Risk volatility between updates

Hourly (`pred`) and event-driven (`load_replay()`), per horizon and segment:
- distribution of |change in risk| between consecutive updates of a patient (median, 90th, 99th percentile);
- threshold flapping for the 24h alert threshold in `policy.json`: crossings per patient-day and the share of
  crossings that reverse within 1, 2 and 3 hours;
- for event-driven updates: how much of the volatility comes from single new measurements (`n_events`).
Figure: |change| percentiles by segment, hourly vs event-driven.

In [ ]:
replay = load_replay()
volatility_table = st.risk_volatility(pred, replay, POLICY["threshold"]["value"], OUT)
del replay


## 5.5 Monotonic consistency across horizons

A patient's risk of AKI within 6h cannot exceed the risk within 12h, and so on. Per snapshot (validation, Platt
and isotonic): share of rows violating 6h <= 12h <= 24h <= 48h for each adjacent pair, size of violations (median and
95th percentile of the excess), and by segment. Use all rows (do not require label eligibility). Describe whether
violations concentrate in particular segments or risk ranges.

In [ ]:
monotonicity_table = st.monotonicity(pred, OUT)


## 5.6 Reliability summary

One table per horizon x segment combining 5.1, 5.4 and 5.5 against the prespecified Demo I criteria (AUROC >= 0.75,
calibration slope and O/E within [0.8, 1.25], >= 100 positives) plus the volatility and monotonicity results, with
a pass/flag column per criterion. Do not change the criteria after seeing the results; report extra criteria you
think are needed separately.

In [ ]:
reliability_table = st.reliability(calibration_table, volatility_table, monotonicity_table, OUT)
print("Prespecified criteria passed:", int(reliability_table.prespecified_all_pass.sum()), "of", len(reliability_table), "horizon-segment cells")


## Hand-back checklist

- [ ] Part 0 passes from a clean kernel.
- [ ] The CSV and PNG files listed at the top are in `artifacts/handoff/jiaju/`; reusable code in `src/aki_eval/stability.py`.
- [ ] A short written summary (5-10 lines) at the end of this notebook: where the model is and is not stable,
      and what that implies for the alert policy and the monitoring window.
- [ ] Notebook outputs cleared of anything patient-level.

## Results and implications

The fixed LightGBM no_dc model was reproduced on 4,950 validation patients and 198,524 snapshots; all four risk and AUROC assertions passed.

Whole-window validation AUROCs (6/12/24/48h): 0.865, 0.839, 0.809, 0.782.

Prespecified reliability criteria passed in 22/36 horizon-segment cells; 6h: 3/9; 12h: 7/9; 24h: 7/9; 48h: 5/9.

Platt segment AUROC ranged 0.705-0.900; calibration slope 0.771-1.246; O/E 0.519-1.364.

Adjacent-segment predictor Spearman ranged 0.992-0.999; top-20 Jaccard 0.667-1.000 (fixed 1,500-patient sample; 100 patient bootstrap draws).

Platt horizon violation rates: 6->12h: 4.38%; 12->24h: 1.65%; 24->48h: 0.83%.

At the locked 24h threshold 0.21, raw crossings were 0.666/patient-day hourly and 0.885/patient-day event-driven (all-update exposure); these are not cooldown-filtered alert counts.

Feature-distribution shifts include time, measurement availability and changing patient composition; SHAP describes associations, not causal drivers.

Flagged cells require qualification when describing reliability. This validation analysis does not reselect the monitoring window, threshold, calibration or model; policy changes need separate project-owner review.

No test data or patient-level outputs are included; uncertainty is based on patient clusters (200 calibration bootstrap draws).

